# Week 6, Thursday

## Course project

Instructions for the course project are posted on Canvas.  Especially important, please keep things manageable and understandable (by you and by me), and use concepts from our course (and potentially 220A/290A).

There is a "Final Project Preparation" homework assignment due Friday of Week 8 (Nov 20).  Part of that assignment is meeting with Luke during his office hours for approximately 10 minutes to discuss your idea for the project or to ask questions / get ideas.  Plan to have that meeting sometime before the due date of that Preparation assignment.

## Evaluating a Neural Network by hand

* What is the difference between "evaluating" a Neural Network and "fitting" a Neural Network?

Here by evaluating, I mean "plug in some inputs, get some outputs".  By fitting, I mean attempt to find what coefficients work well.  (Fitting is done using a variant of gradient descent that we'll talk about later.)

* For the penguins dataset from sckikit-learn, fit a neural network classifier with a single hidden layer of 4 neurons using "bill_length_mm" and "bill_depth_mm" as predictors, and "species" as target.  Use ReLU as our hidden layer activation function, and use "lbfgs" as our solver (my understanding is that this solver is a good choice for relatively small datasets).  Use a random state of 0 and a maximum of 1000 iterations.

In [1]:
import seaborn as sns
import numpy as np
from sklearn.neural_network import MLPClassifier

np.set_printoptions(suppress=True, precision=4) # remove exponential notation

In [2]:
df = sns.load_dataset("penguins")
df = df.dropna(axis=0).copy()

In [3]:
# We convert to NumPy arrays to make rescaling for softmax a little more intuitive
X = df[["bill_length_mm", "bill_depth_mm"]].to_numpy()
y = df["species"].to_numpy()

In [4]:
clf = MLPClassifier(
    hidden_layer_sizes=(4,),
    activation="relu",
    solver="lbfgs",
    random_state=0,
    max_iter=1000
)

In [5]:
clf.fit(X, y)

,"hidden_layer_sizes hidden_layer_sizes: array-like of shape(n_layers - 2,), default=(100,)The ith element represents the number of neurons in the ithhidden layer.","(4,)"
,"solver solver: {'lbfgs', 'sgd', 'adam'}, default='adam'The solver for weight optimization.- 'lbfgs' is an optimizer in the family of quasi-Newton methods.- 'sgd' refers to stochastic gradient descent.- 'adam' refers to a stochastic gradient-based optimizer proposed by Kingma, Diederik, and Jimmy BaFor a comparison between Adam optimizer and SGD, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_training_curves.py`.Note: The default solver 'adam' works pretty well on relativelylarge datasets (with thousands of training samples or more) in terms ofboth training time and validation score.For small datasets, however, 'lbfgs' can converge faster and performbetter.",'lbfgs'
,"max_iter max_iter: int, default=200Maximum number of iterations. The solver iterates until convergence(determined by 'tol') or this number of iterations. For stochasticsolvers ('sgd', 'adam'), note that this determines the number of epochs(how many times each data point will be used), not the number ofgradient steps.",1000
,"random_state random_state: int, RandomState instance, default=NoneDetermines random number generation for weights and biasinitialization, train-test split if early stopping is used, and batchsampling when solver='sgd' or 'adam'.Pass an int for reproducible results across multiple function calls.See :term:`Glossary <random_state>`.",0
,"activation activation: {'identity', 'logistic', 'tanh', 'relu'}, default='relu'Activation function for the hidden layer.- 'identity', no-op activation, useful to implement linear bottleneck, returns f(x) = x- 'logistic', the logistic sigmoid function, returns f(x) = 1 / (1 + exp(-x)).- 'tanh', the hyperbolic tan function, returns f(x) = tanh(x).- 'relu', the rectified linear unit function, returns f(x) = max(0, x)",'relu'
,"alpha alpha: float, default=0.0001Strength of the L2 regularization term. The L2 regularization termis divided by the sample size when added to the loss.For an example usage and visualization of varying regularization, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_alpha.py`.",0.0001
,"batch_size batch_size: int, default='auto'Size of minibatches for stochastic optimizers.If the solver is 'lbfgs', the classifier will not use minibatch.When set to ""auto"", `batch_size=min(200, n_samples)`.",'auto'
,"learning_rate learning_rate: {'constant', 'invscaling', 'adaptive'}, default='constant'Learning rate schedule for weight updates.- 'constant' is a constant learning rate given by 'learning_rate_init'.- 'invscaling' gradually decreases the learning rate at each time step 't' using an inverse scaling exponent of 'power_t'. effective_learning_rate = learning_rate_init / pow(t, power_t)- 'adaptive' keeps the learning rate constant to 'learning_rate_init' as long as training loss keeps decreasing. Each time two consecutive epochs fail to decrease training loss by at least tol, or fail to increase validation score by at least tol if 'early_stopping' is on, the current learning rate is divided by 5.Only used when ``solver='sgd'``.",'constant'
,"learning_rate_init learning_rate_init: float, default=0.001The initial learning rate used. It controls the step-sizein updating the weights. Only used when solver='sgd' or 'adam'.",0.001
,"power_t power_t: float, default=0.5The exponent for inverse scaling learning rate.It is used in updating effective learning rate when the learning_rateis set to 'invscaling'. Only used when solver='sgd'.",0.5
,"shuffle shuffle: bool, default=TrueWhether to shuffle samples in each iteration. Only used whensolver='sgd' or 'adam'.",True


* Try to build up the outputs of this neural network step by step, using the learned coefficients and intercepts.

In [6]:
# which column which species?
clf.classes_

array(['Adelie', 'Chinstrap', 'Gentoo'], dtype='<U9')

In [7]:
X0 = X.copy()

In [ ]:
w0 = clf.coefs_[0]
b0 = clf.intercepts_[0]
# 3 first hidden layer the dim is 2 by 4 
# b_0 dim is 4

In [9]:
# define a relu function

def relu(x):
    return np.maximum(0, x)

In [10]:
X1 = relu(X0@w0 + b0)

In [11]:
w1 = clf.coefs_[1]
b1 = clf.intercepts_[1]

In [13]:
w1.shape

(4, 3)

In [14]:
b1.shape

(3,)

In [ ]:
Z = X1@w1 + b1 # doesn't apply relu here, just scores

In [15]:
Z[:5]

array([[34.7511, 21.7988,  8.2763],
       [29.4875, 24.0427, 14.4101],
       [31.0374, 24.1883, 13.6779],
       [39.1592, 18.1054,  0.723 ],
       [41.7432, 19.4604,  0.984 ]])

Z is sometimes called a score or log odds.  We need to turn it into probabilities.  We will use a hand-defined softmax function (no sigmoid), where $z_i$ gets replaced by $e^{z_i}/\sum_j e^{z_j}$.

The key properties of this strange-looking softmax function are the following.  Notice how these are natural if we want to turn the score vector into a probability vector:
- Preserves the ordering (i.e., is monotone increasing);
- Produces numbers between 0 and 1;
- Produces numbers that sum to 1.

In [17]:
np.exp(Z)[:5]

array([[1.2365e+15, 2.9314e+09, 3.9297e+03],
       [6.4012e+12, 2.7645e+10, 1.8123e+06],
       [3.0156e+13, 3.1976e+10, 8.7148e+05],
       [1.0154e+17, 7.2955e+07, 2.0606e+00],
       [1.3453e+18, 2.8285e+08, 2.6752e+00]])

In [19]:
np.exp(Z)[:5].sum(axis = 1)

array([1.2365e+15, 6.4289e+12, 3.0188e+13, 1.0154e+17, 1.3453e+18])

In [ ]:
# this ir right spiritually but not in NumPuy Syntax
np.exp(Z) / np.exp(Z).sum(axis = 1)

In [ ]:
# probabilities output of softmax
np.exp(Z) / np.exp(Z).sum(axis = 1).reshape(-1, 1)



array([[1.    , 0.    , 0.    ],
       [0.9957, 0.0043, 0.    ],
       [0.9989, 0.0011, 0.    ],
       [1.    , 0.    , 0.    ],
       [1.    , 0.    , 0.    ]])

In [24]:
#  another way
probs = np.exp(Z) / np.exp(Z).sum(axis = 1, keepdims = True)
probs[:5]

array([[1.    , 0.    , 0.    ],
       [0.9957, 0.0043, 0.    ],
       [0.9989, 0.0011, 0.    ],
       [1.    , 0.    , 0.    ],
       [1.    , 0.    , 0.    ]])

In [25]:
clf.predict_proba(X)[:5]

array([[1.    , 0.    , 0.    ],
       [0.9957, 0.0043, 0.    ],
       [0.9989, 0.0011, 0.    ],
       [1.    , 0.    , 0.    ],
       [1.    , 0.    , 0.    ]])

In [26]:
#spiritually correct, but problems with this
# numerical precision issues, will be many trues /falses
clf.predict_proba(X) == (np.exp(Z) / np.exp(Z).sum(axis = 1, keepdims = True))

array([[False,  True, False],
       [False, False,  True],
       [False, False,  True],
       [ True,  True, False],
       [False,  True, False],
       [ True,  True, False],
       [False,  True, False],
       [ True, False,  True],
       [False, False, False],
       [ True, False, False],
       [ True, False, False],
       [False, False, False],
       [False, False, False],
       [ True,  True, False],
       [False,  True, False],
       [False, False, False],
       [False, False, False],
       [ True, False, False],
       [False, False,  True],
       [ True, False,  True],
       [ True,  True, False],
       [False,  True, False],
       [False,  True, False],
       [False, False, False],
       [False, False, False],
       [ True,  True,  True],
       [False, False, False],
       [ True,  True, False],
       [ True, False, False],
       [ True, False, False],
       [ True,  True,  True],
       [False, False, False],
       [ True, False, False],
       [Fa

In [27]:
np.allclose(clf.predict_proba(X), (np.exp(Z) / np.exp(Z).sum(axis = 1, keepdims = True)))

True

## Log loss

We can compute the log loss for our probability estimates using the following.

Reminder: log loss is more refined than accuracy. Log loss has nothing to do with neural networks, works equally welll for any classification problem

In [28]:
from sklearn.metrics import log_loss

In [29]:
log_loss(y, clf.predict_proba(X), labels=clf.classes_)

0.1177449320917818

In [32]:
y[251]

'Gentoo'

In [33]:
clf.predict_proba(X)[251]

array([0.    , 0.0078, 0.9922])

In [34]:
clf.classes_

array(['Adelie', 'Chinstrap', 'Gentoo'], dtype='<U9')

* In a loss function, bigger number is worst
* In Regression we would use MSE or MAE
* In classification we might use log loss

* Can we reproduce the that number from the probabilities directly?  The formula for log loss is $\frac{1}{n} \sum -\log(p_i)$, where $n$ is the number of observations, and where $p_i$ is the predicted probability assigned to the true value of $y_i$.  (As an example of what needs to be done, consider the second-to-last row.)

In [35]:
probs[-2]

array([0.    , 0.0622, 0.9378])

In [36]:
y[-2]

'Gentoo'

In [39]:
class_list = list(clf.classes_)

In [40]:
class_list.index("Adelie")

0

In [41]:
class_list.index("Gentoo")

2

In [44]:
# which column to take for each row?
p_list = []
for label, p_row in zip(y,probs):
    ind = class_list.index(label)
    p_list.append(p_row[ind])

for label,p_row in zip(y, probs):
    print(label)
    print(p_row)
    break

Adelie
[1. 0. 0.]


In [45]:
len(p_list)

333

In [47]:
(-np.log(p_list)).mean()

np.float64(0.1177449320917818)

## Review

Some topics.  Use lectures, homework, and sample midterm to get a sense of topics that might appear.

* Data types, with advantages, disadvantages - short answer

* Indexing (especially Boolean indexing) - write out the code

* Plotting with Matplotlib and Seaborn -  short answer

* Random numbers and probability estimates - write out the code

* Linear regression and predictions - write out the code / here are the coefs what are the predictions you make?

* Logistic regression and predictions, sigmoid - write out the code / sigmoid of x and how it realtes to function

* Loss functions (including log loss) - short answer

* Overfitting - short answer

* Bias variance decomposition - short answer

* Neural networks and predictions - code or short answer



In [ ]:
# what does plot9([1,3,5],[2,4,6]) look like
